# Class-Conditional Image Generation using a Conditional GAN (cGAN)

**Course:** Machine Learning / Deep Learning Assignment  
**Dataset:** MNIST Handwritten Digits (28x28 grayscale)  
**Framework:** PyTorch & Torchvision  

---

## 1. Project Objectives
1. Build and train a **Conditional Generative Adversarial Network (cGAN)** on MNIST.
2. Condition image generation on target digit class labels ($0$ through $9$).
3. Demonstrate that providing class labels guides the generated output while latent noise provides sample variation.
4. Analyze the real training dynamics and loss curves recorded across 10 epochs.

## 2. Setup & Module Imports
We import standard scientific libraries and our modular project classes from `src/`.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in sys.path so modules in src/ can be imported
project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.model import Generator
from src.generate import load_generator, generate_single_class_samples

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")
print(f"Project root: {project_root}")

## 3. Load Trained Model Checkpoint
We load the trained weights from `checkpoints/generator_checkpoint.pth` (saved at Epoch 10).

In [ ]:
checkpoint_path = project_root / "checkpoints" / "generator_checkpoint.pth"

if not checkpoint_path.exists():
    print("[!] Checkpoint not found. Please run 'python src/train.py --epochs 10' first.")
else:
    generator, latent_dim, num_classes, epoch = load_generator(str(checkpoint_path), device=device)
    print(f"Successfully loaded checkpoint from Epoch {epoch}.")

## 4. Interactive Digit Generation
Select any digit class label ($0$ to $9$) below to generate 16 diverse samples.

In [ ]:
target_digit = 7  # Change this to any digit from 0 to 9
num_samples = 16

# Sample random latent vectors
noise = torch.randn(num_samples, latent_dim, device=device)
labels = torch.full((num_samples,), fill_value=target_digit, dtype=torch.int64, device=device)

with torch.no_grad():
    generated_tensors = generator(noise, labels)
    # Denormalize from [-1, 1] to [0, 1]
    images = (generated_tensors * 0.5) + 0.5
    images = torch.clamp(images, 0.0, 1.0).cpu()

# Plot in 4x4 grid
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
fig.suptitle(f"Generated Handwritten Digit Samples: '{target_digit}'", fontsize=13, fontweight="bold")

for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(0).numpy(), cmap="gray")
    ax.set_title(f"#{i+1}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Condition Comparison Grid (Isolating Class Labels)
In this experiment, we feed the **same noise vector** across all 10 digit conditions.
This isolates the class label variable to evaluate whether the condition controls output shape.

In [ ]:
comparison_path = project_root / "results" / "condition_comparison.png"

if comparison_path.exists():
    img = Image.open(comparison_path)
    plt.figure(figsize=(14, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Condition Comparison Grid (Columns: Class 0-9 | Rows: Same Noise Vector)", fontsize=12, fontweight="bold")
    plt.show()
else:
    print("[!] Comparison grid not found. Run 'python src/generate.py' to generate it.")

## 6. Training Loss Dynamics & Adversarial Balance
We plot the actual recorded Generator and Discriminator losses across the 10 training epochs.

In [ ]:
plot_path = project_root / "results" / "training_loss_plot.png"

if plot_path.exists():
    img = Image.open(plot_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Recorded cGAN Loss Curves (10 Epochs on MNIST)", fontsize=12, fontweight="bold")
    plt.show()
else:
    print("[!] Loss plot not found. Run 'python src/analyze_results.py' to create it.")

## 7. Analysis & Discussion

### Key Findings
1. **Loss Dynamics:** The Discriminator's loss dropped to near-zero ($0.0003$) while Generator loss climbed to $12.01$. This reflects **Discriminator dominance**, a well-documented phenomenon where the Discriminator learns faster than the Generator, causing diminishing gradient feedback.
2. **Visual Assessment:** The Generator produces visible central stroke structures. However, speckle noise artifacts and limited separation between complex digits (e.g. $8$ vs $3$) illustrate the limitations of a flat MLP architecture without spatial convolutions.

### Recommended Future Improvements
- **Deep Convolutional GAN (DCGAN):** Replace `Linear` layers with `Conv2d` and `ConvTranspose2d` to capture spatial 2D pixel correlations.
- **One-Sided Label Smoothing:** Set real targets to $0.9$ instead of $1.0$ to soften the Discriminator's confidence.
- **Condition Embedding:** Use learned `nn.Embedding` representations instead of small one-hot vectors.